# Semivolatility-Scaled Momentum with Historical Membership

Uses a moving-average crossover and Garman–Klass semivolatility to rank and size a monthly stock portfolio.

Set `SMOM_MEMBERSHIP_CSV` to the path of a local CSV with the columns `symbol,start_date,end_date`, using Yahoo Finance symbols and ISO dates. Each row describes one membership interval. The start is inclusive and the end is exclusive. Leave the end blank for ongoing membership. The notebook screens the portfolio using membership on each signal date. Use complete historical membership for the full test period. Yahoo Finance may lack prices for delisted names or historical ticker mappings, and those gaps can prevent a faithful full-universe run. The notebook requires observed prices while a symbol is held.

In [ ]:
"""
Semi-Volatility-Scaled Momentum Strategy - PRODUCTION VERSION
==============================================================
Momentum Signal: SMA 20-200 Crossover
Volatility Measure: Garman-Klass Semi-Volatility (downside only)

CRITICAL FIXES IMPLEMENTED:
1. Point-in-time universe selection (no survivorship/look-ahead bias)
2. Expanding window semi-volatility (no future data leakage)
3. Correct semi-variance calculation (divide by negative days only)
4. Proper rebalancing chronology (signal at Close[t], execute at Open[t+1])
5. Asset ranking by momentum strength for position selection
6. Adjusted prices (handles splits/dividends)
7. Leverage constraints applied
8. All returns calculated correctly including last day

Trading Flow:
- Signal generated at Close[t]
- Execution at Open[t+1]
- Portfolio valued at Open[t+1]
- Returns = Open[t+2]/Open[t+1] - 1 on rebalancing days
- During holding: Open[t+1]/Open[t] - 1 (consistent with rebalancing logic)
"""

import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from _market_data import download_ohlcv
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

class StrategyConfig:
    """
    Centralized configuration for strategy parameters.
    All parameters are easily adjustable here.
    """
    MEMBERSHIP_CSV = os.environ.get("SMOM_MEMBERSHIP_CSV")

    # Date Range
    END_DATE = "2024-12-31"
    START_DATE = "2019-01-01"  # Start date for backtest (need buffer for indicators)
    WARMUP_DATE = "2018-01-01"  # Additional warmup for 200-day SMA calculation

    # Momentum Parameters
    SMA_SHORT = 20      # Short-term SMA for crossover
    SMA_LONG = 200      # Long-term SMA for crossover

    # Volatility Parameters
    SEMI_VOL_WINDOW = 60        # Lookback window for semi-volatility calculation (in days)
    TARGET_VOLATILITY = 0.15    # Target annual volatility (15%)
    ANNUALIZATION_FACTOR = 252  # Trading days per year

    # Transaction Costs
    COMMISSION_BPS = 10         # Commission in basis points (0.10%)
    SLIPPAGE_BPS = 5            # Slippage in basis points (0.05%)
    TOTAL_COST_BPS = COMMISSION_BPS + SLIPPAGE_BPS  # Total cost per trade

    # Portfolio Parameters
    INITIAL_CAPITAL = 100000    # Starting capital in USD
    MAX_POSITIONS = 20          # Maximum number of concurrent positions
    MIN_POSITION_SIZE = 0.01    # Minimum position size (1% of portfolio)

    # Rebalancing
    REBALANCE_FREQUENCY = 'M'   # 'M' for monthly, 'Q' for quarterly

    # Benchmark
    BENCHMARK_TICKER = "SPY"

    # Risk Management
    MAX_LEVERAGE = 1.0          # Maximum gross leverage (1.0 = no leverage)

    # Universe Selection
    MIN_MARKET_CAP = 2e9        # $2B minimum market cap
    MIN_PRICE = 5.0             # $5 minimum price (avoid penny stocks)
    MIN_VOLUME = 1e6            # 1M daily volume minimum


# ============================================================================
# DATA ACQUISITION MODULE - FIXED FOR POINT-IN-TIME UNIVERSE
# ============================================================================

class YahooDataFetcher:
    """Load dividend-adjusted daily OHLCV from Yahoo Finance."""

    @staticmethod
    def get_historical_data(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        prices = download_ohlcv(ticker, start_date, end_date, auto_adjust=True)
        prices = prices.rename(columns=str.lower)
        columns = ["open", "high", "low", "close"]
        if (prices[columns] <= 0).any().any():
            raise ValueError(f"Yahoo Finance returned non-positive prices for {ticker}.")
        return prices[columns + ["volume"]]


def read_membership(path, start_date, end_date):
    """Load ticker membership intervals with inclusive start and exclusive end."""
    if not path:
        raise ValueError("Set SMOM_MEMBERSHIP_CSV to a local constituent history CSV.")
    membership = pd.read_csv(path, usecols=["symbol", "start_date", "end_date"])
    membership["symbol"] = membership["symbol"].astype("string").str.strip().str.upper()
    if not membership["symbol"].str.fullmatch(r"[A-Z0-9][A-Z0-9.-]{0,15}").fillna(False).all():
        raise ValueError("The membership file contains an invalid Yahoo Finance symbol.")
    for field in ("start_date", "end_date"):
        membership[field] = pd.to_datetime(membership[field], format="%Y-%m-%d")
    if membership["start_date"].isna().any():
        raise ValueError("Every membership interval needs a start date.")
    without_end = membership["end_date"].isna()
    if ((~without_end) & (membership["end_date"] <= membership["start_date"])).any():
        raise ValueError("Each membership end date must be later than its start date.")
    overlaps = ((membership["start_date"] <= pd.Timestamp(end_date)) &
                (without_end | (membership["end_date"] > pd.Timestamp(start_date))))
    membership = membership.loc[overlaps]
    if membership.empty:
        raise ValueError("No membership intervals overlap the backtest period.")
    return membership


def apply_membership(signals, membership):
    allowed = pd.DataFrame(False, index=signals.index, columns=signals.columns)
    for row in membership.itertuples(index=False):
        dates = allowed.index >= row.start_date
        if pd.notna(row.end_date):
            dates &= allowed.index < row.end_date
        if row.symbol in allowed:
            allowed.loc[dates, row.symbol] = True
    return signals.where(allowed, 0)


class GarmanKlassSemiVolatility:
    """
    Implements Garman-Klass volatility estimator adapted for semi-volatility.

    CRITICAL FIXES:
    1. Uses expanding window (not rolling) to avoid look-ahead bias
    2. Correctly divides by count of negative days (not total window size)
    3. Handles negative variance properly (max with 0)
    4. No global median filling (uses expanding calculation only)
    """

    @staticmethod
    def calculate(df: pd.DataFrame, window: int, annualization_factor: int = 252) -> pd.Series:
        """
        Calculate expanding Garman-Klass semi-volatility with no look-ahead bias.

        ALGORITHM (NO FUTURE DATA):
        1. For each date t, look back at previous 'window' days only
        2. Calculate GK variance for each day in lookback
        3. Keep only days with negative close-to-close returns
        4. Average variance across negative days ONLY (divide by count of negative days)
        5. Annualize the result

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with 'open', 'high', 'low', 'close' columns
        window : int
            Lookback window size (in days) - maximum history to consider
        annualization_factor : int
            Number of trading days per year (typically 252)

        Returns:
        --------
        pd.Series : Annualized Garman-Klass semi-volatility (no future data used)
        """
        # Calculate log returns for OHLC
        log_hl = np.log(df['high'] / df['low'])
        log_co = np.log(df['close'] / df['open'])

        # Standard Garman-Klass variance (daily)
        # Formula: 0.5 * (log(H/L))^2 - (2*log(2)-1) * (log(C/O))^2
        gk_variance = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)

        # CRITICAL FIX: Handle negative variance from GK formula
        # The GK formula can produce negative values due to the negative term
        # Take max(0, variance) to ensure non-negative variance
        gk_variance = np.maximum(gk_variance, 0)

        # Calculate close-to-close returns to identify negative days
        # Shift(1) ensures we compare today's close to yesterday's close
        close_returns = df['close'].pct_change()

        # Create indicator for negative return days (1 if negative, 0 if positive)
        is_negative_day = (close_returns < 0).astype(int)

        # Initialize semi-volatility series
        semi_vol = pd.Series(np.nan, index=df.index)

        # CRITICAL FIX: Calculate semi-volatility using only past data (expanding window)
        for i in range(window, len(df)):
            # Look back exactly 'window' days (including current day)
            lookback_start = max(0, i - window + 1)
            lookback_slice = slice(lookback_start, i + 1)

            # Get variance and negative day indicator for this lookback period
            variance_window = gk_variance.iloc[lookback_slice]
            negative_window = is_negative_day.iloc[lookback_slice]

            # CRITICAL FIX: Only include variance from negative return days
            negative_variance = variance_window[negative_window == 1]

            # Count of negative days in window
            n_negative_days = len(negative_variance)

            # Calculate average semi-variance (only over negative days)
            if n_negative_days > 0:
                # CORRECT CALCULATION: sum of variances / count of negative days
                avg_semi_variance = negative_variance.sum() / n_negative_days
            else:
                # If no negative days in window, use the previous semi-vol value
                # This avoids division by zero and doesn't inject future information
                if i > window:
                    avg_semi_variance = (semi_vol.iloc[i-1] / np.sqrt(annualization_factor)) ** 2
                else:
                    # For very first calculation with no negative days, use a conservative estimate
                    avg_semi_variance = gk_variance.iloc[lookback_slice].mean()

            # Convert variance to volatility (standard deviation)
            # Annualize by multiplying by sqrt(trading days per year)
            semi_vol.iloc[i] = np.sqrt(avg_semi_variance * annualization_factor)

        # For the initial period (before we have 'window' days), use expanding window
        for i in range(1, min(window, len(df))):
            lookback_slice = slice(0, i)

            variance_window = gk_variance.iloc[lookback_slice]
            negative_window = is_negative_day.iloc[lookback_slice]

            negative_variance = variance_window[negative_window == 1]
            n_negative_days = len(negative_variance)

            if n_negative_days > 0:
                avg_semi_variance = negative_variance.sum() / n_negative_days
                semi_vol.iloc[i] = np.sqrt(avg_semi_variance * annualization_factor)
            elif i > 1:
                semi_vol.iloc[i] = semi_vol.iloc[i-1]

        return semi_vol


# ============================================================================
# MOMENTUM SIGNAL GENERATOR
# ============================================================================

class SMAMomentumSignal:
    """
    Generates momentum signals based on Simple Moving Average crossover.

    Signal Logic:
    - LONG (1): When SMA_short > SMA_long (bullish momentum)
    - NEUTRAL (0): When SMA_short <= SMA_long (bearish momentum)

    Also calculates signal strength for ranking assets.
    """

    @staticmethod
    def calculate(df: pd.DataFrame, short_window: int, long_window: int) -> Tuple[pd.Series, pd.Series]:
        """
        Calculate SMA crossover signals and signal strength.

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with 'close' column
        short_window : int
            Short-term SMA period (e.g., 20 days)
        long_window : int
            Long-term SMA period (e.g., 200 days)

        Returns:
        --------
        Tuple[pd.Series, pd.Series] : (binary signal, signal strength)
            - signal: 1 = long, 0 = cash
            - strength: % difference between short and long SMA (momentum intensity)
        """
        # Calculate both moving averages
        sma_short = df['close'].rolling(window=short_window, min_periods=short_window).mean()
        sma_long = df['close'].rolling(window=long_window, min_periods=long_window).mean()

        # Generate binary signal: 1 when short MA above long MA, 0 otherwise
        signal = (sma_short > sma_long).astype(int)

        # CRITICAL FIX: Calculate signal strength for ranking
        # Strength = (SMA_short - SMA_long) / SMA_long
        # Higher positive values = stronger bullish momentum
        signal_strength = (sma_short - sma_long) / sma_long
        signal_strength = signal_strength.fillna(0)

        # CRITICAL: Shift signal by 1 to avoid look-ahead bias
        # Signal generated at Close[t] is used for trading at Open[t+1]
        signal = signal.shift(1)
        signal_strength = signal_strength.shift(1)

        return signal, signal_strength


# ============================================================================
# POSITION SIZING ENGINE - FIXED
# ============================================================================

class VolatilityScaledPositionSizer:
    """
    Calculates position sizes based on inverse semi-volatility weighting.

    CRITICAL FIXES:
    1. Ranks assets by signal strength before selecting top N
    2. Applies MAX_LEVERAGE constraint
    3. Ensures weights are calculated using data available at time t
    """

    @staticmethod
    def calculate_weights(signals: pd.DataFrame,
                         signal_strengths: pd.DataFrame,
                         semi_vols: pd.DataFrame,
                         target_vol: float,
                         max_positions: int,
                         max_leverage: float) -> pd.DataFrame:
        """
        Calculate position weights using semi-volatility scaling with proper asset ranking.

        ALGORITHM:
        1. For each date, identify assets with positive signal (binary=1)
        2. Rank these assets by signal strength (momentum intensity)
        3. Select top MAX_POSITIONS ranked assets
        4. Calculate raw weight = (Target_Vol / Semi_Vol_i) for each selected asset
        5. Normalize weights to respect MAX_LEVERAGE constraint
        6. Apply minimum position size filter

        Parameters:
        -----------
        signals : pd.DataFrame
            Binary signals (1/0) for each asset
        signal_strengths : pd.DataFrame
            Signal strength (momentum intensity) for ranking
        semi_vols : pd.DataFrame
            Semi-volatility for each asset
        target_vol : float
            Target portfolio volatility (annualized)
        max_positions : int
            Maximum number of positions
        max_leverage : float
            Maximum gross leverage (e.g., 1.0 = fully invested, no leverage)

        Returns:
        --------
        pd.DataFrame : Position weights (% of portfolio)
        """
        # Initialize weights DataFrame
        weights = pd.DataFrame(0.0, index=signals.index, columns=signals.columns)

        # Iterate through each date
        for date in signals.index:
            # Get active signals (assets with signal = 1)
            active_signals = signals.loc[date]
            active_mask = active_signals == 1

            if not active_mask.any():
                continue  # No positions on this date

            # Get signal strengths for active assets
            strengths = signal_strengths.loc[date, active_mask]

            # CRITICAL FIX: Rank by signal strength and select top MAX_POSITIONS
            # Higher strength = stronger momentum = higher priority
            ranked_tickers = strengths.sort_values(ascending=False).head(max_positions).index.tolist()

            if len(ranked_tickers) == 0:
                continue

            # Get semi-volatilities for selected assets
            asset_semi_vols = semi_vols.loc[date, ranked_tickers]

            # Skip if any semi-vol is NaN or zero
            valid_mask = (asset_semi_vols.notna()) & (asset_semi_vols > 0)
            if not valid_mask.any():
                continue

            ranked_tickers = asset_semi_vols[valid_mask].index.tolist()
            asset_semi_vols = asset_semi_vols[valid_mask]

            # Calculate inverse volatility weights
            # Raw weight ∝ 1 / Semi_Vol
            inverse_vols = target_vol / asset_semi_vols

            # Normalize to sum to MAX_LEVERAGE
            raw_weights = inverse_vols / inverse_vols.sum() * max_leverage

            # Apply minimum position size constraint
            raw_weights = raw_weights[raw_weights >= StrategyConfig.MIN_POSITION_SIZE]

            # Re-normalize after filtering
            if raw_weights.sum() > 0:
                final_weights = raw_weights / raw_weights.sum() * max_leverage
                weights.loc[date, final_weights.index] = final_weights.values

        return weights


# ============================================================================
# BACKTESTING ENGINE - FIXED
# ============================================================================

class SMOMBacktester:
    """
    Comprehensive backtesting engine with all critical fixes applied.

    CRITICAL FIXES:
    1. Point-in-time universe (no survivorship bias)
    2. Proper rebalancing timing (signal Close[t] → execute Open[t+1])
    3. Consistent return calculation (Open-to-Open for all periods)
    4. Correct last day handling
    5. All leverage constraints applied
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.data_fetcher = YahooDataFetcher()

    def run_backtest(self) -> Dict:
        """
        Execute complete backtest workflow with all fixes applied.

        Returns:
        --------
        Dict : Contains all backtest results, performance metrics, and data
        """
        print("="*80)
        print("SEMIVOLATILITY-SCALED MOMENTUM WITH HISTORICAL MEMBERSHIP")
        print("="*80)
        print(f"Period: {self.config.START_DATE} to {self.config.END_DATE}")
        print(f"Warmup: {self.config.WARMUP_DATE} (for {self.config.SMA_LONG}-day SMA)")
        print(f"Momentum: SMA {self.config.SMA_SHORT}-{self.config.SMA_LONG} Crossover")
        print(f"Semi-Volatility: Garman-Klass {self.config.SEMI_VOL_WINDOW}-day expanding window")
        print(f"Target Volatility: {self.config.TARGET_VOLATILITY*100:.1f}%")
        print(f"Max Leverage: {self.config.MAX_LEVERAGE:.1f}x")
        print(f"Transaction Costs: {self.config.TOTAL_COST_BPS} bps per trade")
        print("="*80)

        # Step 1: Get point-in-time universe (S&P 500)
        print("\n[1/7] Establishing point-in-time universe...")
        membership = read_membership(self.config.MEMBERSHIP_CSV, self.config.START_DATE, self.config.END_DATE)
        tickers = membership['symbol'].drop_duplicates().tolist()

        if len(tickers) == 0:
            raise ValueError("No tickers retrieved")

        print(f"Universe: {len(tickers)} tickers")

        # Step 2: Download historical data
        print("\n[2/7] Downloading adjusted historical data...")
        benchmark_data = self.data_fetcher.get_historical_data(
            self.config.BENCHMARK_TICKER, self.config.WARMUP_DATE, self.config.END_DATE,
        )
        price_data = {}
        failed_tickers = []

        for i, ticker in enumerate(tickers):
            if (i + 1) % 20 == 0:
                print(f"  Progress: {i+1}/{len(tickers)} tickers processed")

            data = self.data_fetcher.get_historical_data(
                ticker,
                self.config.WARMUP_DATE,  # Include warmup period
                self.config.END_DATE
            )

            # Minimum data requirement: enough for long SMA
            if data is not None and len(data) >= self.config.SMA_LONG:
                price_data[ticker] = data
            else:
                failed_tickers.append(ticker)

        print(f"\nSuccessfully loaded {len(price_data)} tickers")
        print(f"Failed/insufficient data: {len(failed_tickers)} tickers")

        if len(price_data) == 0:
            raise ValueError("No valid price data retrieved")

        # Step 3: Calculate Momentum Signals and Strength
        print("\n[3/7] Calculating momentum signals and ranking strength...")
        signals = pd.DataFrame(index=benchmark_data.index)
        signal_strengths = pd.DataFrame(index=benchmark_data.index)

        for ticker, data in price_data.items():
            signal, strength = SMAMomentumSignal.calculate(
                data,
                self.config.SMA_SHORT,
                self.config.SMA_LONG
            )
            signals[ticker] = signal
            signal_strengths[ticker] = strength

        # Step 4: Calculate Semi-Volatility with expanding window
        print("\n[4/7] Computing Garman-Klass semi-volatility (expanding window)...")
        semi_vols = pd.DataFrame(index=benchmark_data.index)

        for ticker, data in price_data.items():
            semi_vol = GarmanKlassSemiVolatility.calculate(
                data,
                self.config.SEMI_VOL_WINDOW,
                self.config.ANNUALIZATION_FACTOR
            )
            semi_vols[ticker] = semi_vol

        # Step 5: Align all data to backtest period (exclude warmup)
        print("\n[5/7] Aligning data to backtest period...")
        backtest_start = pd.to_datetime(self.config.START_DATE)
        backtest_end = pd.to_datetime(self.config.END_DATE)

        common_dates = signals.index.intersection(semi_vols.index)
        common_dates = common_dates[(common_dates >= backtest_start) & (common_dates <= backtest_end)]

        if common_dates.empty or not price_data:
            raise ValueError("No eligible price data in the backtest period.")
        signals = apply_membership(signals.loc[common_dates], membership)
        signal_strengths = signal_strengths.loc[common_dates]
        semi_vols = semi_vols.loc[common_dates]

        print(f"Backtest period: {common_dates[0].strftime('%Y-%m-%d')} to {common_dates[-1].strftime('%Y-%m-%d')}")
        print(f"Total trading days: {len(common_dates)}")

        # Step 6: Calculate Position Weights with ranking
        print("\n[6/7] Calculating position weights with momentum ranking...")
        weights = VolatilityScaledPositionSizer.calculate_weights(
            signals,
            signal_strengths,
            semi_vols,
            self.config.TARGET_VOLATILITY,
            self.config.MAX_POSITIONS,
            self.config.MAX_LEVERAGE
        )

        # Step 7: Simulate Trading with proper timing
        print("\n[7/7] Simulating strategy execution...")

        # Extract open and close prices
        open_prices = pd.DataFrame({ticker: data['open'] for ticker, data in price_data.items()})
        close_prices = pd.DataFrame({ticker: data['close'] for ticker, data in price_data.items()})

        # Align to backtest period
        open_prices = open_prices.reindex(common_dates)
        close_prices = close_prices.reindex(common_dates)

        # Calculate strategy returns with fixed timing
        strategy_returns = self._calculate_strategy_returns_fixed(
            weights,
            open_prices,
            close_prices
        )

        # Get Benchmark Returns
        print("\nLoading benchmark data...")

        # Benchmark returns: Open-to-Open (consistent with strategy)
        benchmark_open = benchmark_data['open'].loc[common_dates]
        benchmark_returns = benchmark_open.pct_change()

        # Calculate performance metrics
        results = self._calculate_performance_metrics(
            strategy_returns,
            benchmark_returns
        )

        # Store additional data
        results['weights'] = weights
        results['signals'] = signals
        results['signal_strengths'] = signal_strengths
        results['semi_vols'] = semi_vols
        results['price_data'] = price_data

        print("\n" + "="*80)
        print("BACKTEST COMPLETE")
        print("="*80)

        return results

    def _calculate_strategy_returns_fixed(self,
                                          weights: pd.DataFrame,
                                          open_prices: pd.DataFrame,
                                          close_prices: pd.DataFrame) -> pd.Series:
        """
        Calculate strategy returns with proper timing and no look-ahead bias.

        CORRECTED TIMING:
        - Monthly rebalancing: Signal at last Close of month, execute at first Open of next month
        - All returns calculated Open[t] to Open[t+1] consistently
        - Transaction costs applied on rebalancing days
        - Last day handled correctly

        Parameters:
        -----------
        weights : pd.DataFrame
            Target position weights (daily)
        open_prices : pd.DataFrame
            Open prices for all assets
        close_prices : pd.DataFrame
            Close prices for all assets

        Returns:
        --------
        pd.Series : Daily strategy returns
        """
        # Identify month-end dates for rebalancing
        dates = weights.index
        month_ends = dates.to_series().groupby(dates.to_period('M')).apply(lambda x: x.index[-1])

        # Rebalancing happens at open of NEXT trading day after month-end
        rebalance_dates = []
        for month_end in month_ends:
            month_end_idx = dates.get_loc(month_end)
            if month_end_idx < len(dates) - 1:
                next_trading_day = dates[month_end_idx + 1]
                rebalance_dates.append(next_trading_day)

        rebalance_dates = pd.DatetimeIndex(rebalance_dates)

        # Get weights at month-end (these will be executed at open of next day)
        rebalancing_weights = weights.loc[month_ends]

        # Initialize returns and position tracking
        strategy_returns = pd.Series(0.0, index=dates)
        current_weights = pd.Series(0.0, index=weights.columns)

        # Track transaction costs
        transaction_costs_ts = pd.Series(0.0, index=dates)

        # Calculate returns day by day
        for i in range(len(dates)):
            date = dates[i]

            # Check if this is a rebalancing day
            if date in rebalance_dates:
                # Find which month-end triggered this rebalancing
                month_end = month_ends[month_ends < date].iloc[-1]
                new_weights = weights.loc[month_end]

                # Calculate turnover (sum of absolute weight changes)
                turnover = (new_weights - current_weights).abs().sum()

                # Apply transaction costs
                transaction_costs_ts.loc[date] = turnover * self.config.TOTAL_COST_BPS / 10000

                # Update current weights
                current_weights = new_weights

            # Calculate returns for this day: Open[t] to Open[t+1]
            if i < len(dates) - 1:
                next_date = dates[i + 1]

                # Asset returns: Open[t+1] / Open[t] - 1
                asset_returns = (open_prices.loc[next_date] / open_prices.loc[date] - 1)

                # Portfolio return: weighted sum
                missing = asset_returns.isna() & current_weights.ne(0)
                if missing.any():
                    symbols = ", ".join(missing.index[missing])
                    raise ValueError(f"Missing prices for held symbols on {date.date()}: {symbols}")
                portfolio_return = (current_weights * asset_returns).sum()

                # Apply transaction costs if rebalancing day
                net_return = portfolio_return - transaction_costs_ts.loc[date]

                strategy_returns.loc[date] = net_return
            else:
                # CRITICAL FIX: Last day calculation
                # For the last day, we can't calculate Open[t+1] return
                # Use Close[t] / Open[t] - 1 as the final period return
                asset_returns = (close_prices.loc[date] / open_prices.loc[date] - 1)
                missing = asset_returns.isna() & current_weights.ne(0)
                if missing.any():
                    symbols = ", ".join(missing.index[missing])
                    raise ValueError(f"Missing prices for held symbols on {date.date()}: {symbols}")
                portfolio_return = (current_weights * asset_returns).sum()
                strategy_returns.loc[date] = portfolio_return

        return strategy_returns

    def _calculate_performance_metrics(self,
                                       strategy_returns: pd.Series,
                                       benchmark_returns: pd.Series) -> Dict:
        """
        Calculate comprehensive performance metrics.

        Parameters:
        -----------
        strategy_returns : pd.Series
            Strategy daily returns
        benchmark_returns : pd.Series
            Benchmark daily returns

        Returns:
        --------
        Dict : Performance metrics and statistics
        """
        # Align returns
        common_dates = strategy_returns.index.intersection(benchmark_returns.index)
        strat_ret = strategy_returns.loc[common_dates]
        bench_ret = benchmark_returns.loc[common_dates]

        # Remove first day (NaN from pct_change)
        strat_ret = strat_ret.dropna()
        bench_ret = bench_ret.dropna()

        # Align again after dropping NaNs
        common_dates = strat_ret.index.intersection(bench_ret.index)
        strat_ret = strat_ret.loc[common_dates]
        bench_ret = bench_ret.loc[common_dates]

        # Calculate cumulative returns
        strat_cum = (1 + strat_ret).cumprod()
        bench_cum = (1 + bench_ret).cumprod()

        # Annualization
        ann_factor = self.config.ANNUALIZATION_FACTOR
        years = len(strat_ret) / ann_factor

        # Performance Metrics
        metrics = {
            # Returns
            'total_return_strategy': strat_cum.iloc[-1] - 1,
            'total_return_benchmark': bench_cum.iloc[-1] - 1,
            'cagr_strategy': (strat_cum.iloc[-1] ** (1/years)) - 1,
            'cagr_benchmark': (bench_cum.iloc[-1] ** (1/years)) - 1,

            # Volatility
            'volatility_strategy': strat_ret.std() * np.sqrt(ann_factor),
            'volatility_benchmark': bench_ret.std() * np.sqrt(ann_factor),

            # Downside Risk
            'downside_dev_strategy': strat_ret[strat_ret < 0].std() * np.sqrt(ann_factor),
            'downside_dev_benchmark': bench_ret[bench_ret < 0].std() * np.sqrt(ann_factor),

            # Risk-Adjusted Returns
            'sharpe_strategy': (strat_ret.mean() * ann_factor) / (strat_ret.std() * np.sqrt(ann_factor)) if strat_ret.std() > 0 else 0,
            'sharpe_benchmark': (bench_ret.mean() * ann_factor) / (bench_ret.std() * np.sqrt(ann_factor)) if bench_ret.std() > 0 else 0,

            # Sortino Ratio
            'sortino_strategy': (strat_ret.mean() * ann_factor) / (strat_ret[strat_ret < 0].std() * np.sqrt(ann_factor)) if len(strat_ret[strat_ret < 0]) > 0 else 0,
            'sortino_benchmark': (bench_ret.mean() * ann_factor) / (bench_ret[bench_ret < 0].std() * np.sqrt(ann_factor)) if len(bench_ret[bench_ret < 0]) > 0 else 0,

            # Drawdowns
            'max_drawdown_strategy': self._calculate_max_drawdown(strat_cum),
            'max_drawdown_benchmark': self._calculate_max_drawdown(bench_cum),

            # Win Rates
            'win_rate_strategy': (strat_ret > 0).sum() / len(strat_ret),
            'win_rate_benchmark': (bench_ret > 0).sum() / len(bench_ret),

            # Time Series
            'strategy_returns': strat_ret,
            'benchmark_returns': bench_ret,
            'strategy_cumulative': strat_cum,
            'benchmark_cumulative': bench_cum,
        }

        return metrics

    @staticmethod
    def _calculate_max_drawdown(cumulative_returns: pd.Series) -> float:
        """Calculate maximum drawdown from cumulative returns."""
        running_max = cumulative_returns.expanding().max()
        drawdown = (cumulative_returns - running_max) / running_max
        return drawdown.min()


# ============================================================================
# VISUALIZATION MODULE
# ============================================================================

class PerformanceVisualizer:
    """
    Creates publication-quality visualizations for strategy performance.
    """

    @staticmethod
    def plot_results(results: Dict, config: StrategyConfig):
        """Generate comprehensive performance visualization dashboard."""
        fig = plt.figure(figsize=(20, 12))
        gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

        # Extract data
        strat_cum = results['strategy_cumulative']
        bench_cum = results['benchmark_cumulative']
        strat_ret = results['strategy_returns']
        bench_ret = results['benchmark_returns']

        # 1. Cumulative Returns
        ax1 = fig.add_subplot(gs[0, :])
        ax1.plot(strat_cum.index, strat_cum.values,
                label='Strategy', linewidth=2, color='#2E86AB')
        ax1.plot(bench_cum.index, bench_cum.values,
                label=f'Benchmark ({config.BENCHMARK_TICKER})',
                linewidth=2, color='#A23B72', linestyle='--')
        ax1.set_title('Cumulative Returns', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Cumulative Return', fontsize=12)
        ax1.legend(loc='upper left', fontsize=10)
        ax1.grid(True, alpha=0.3)
        ax1.axhline(y=1, color='black', linestyle='-', linewidth=0.5)

        # 2. Rolling Sharpe
        ax2 = fig.add_subplot(gs[1, 0])
        rolling_window = 252
        rolling_sharpe_strat = (
            strat_ret.rolling(rolling_window).mean() /
            strat_ret.rolling(rolling_window).std() *
            np.sqrt(config.ANNUALIZATION_FACTOR)
        )
        rolling_sharpe_bench = (
            bench_ret.rolling(rolling_window).mean() /
            bench_ret.rolling(rolling_window).std() *
            np.sqrt(config.ANNUALIZATION_FACTOR)
        )
        ax2.plot(rolling_sharpe_strat.index, rolling_sharpe_strat.values,
                label='Strategy', linewidth=1.5, color='#2E86AB')
        ax2.plot(rolling_sharpe_bench.index, rolling_sharpe_bench.values,
                label='Benchmark', linewidth=1.5, color='#A23B72', linestyle='--')
        ax2.set_title(f'Rolling Sharpe Ratio ({rolling_window}d)', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Sharpe Ratio', fontsize=10)
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3)
        ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

        # 3. Drawdown
        ax3 = fig.add_subplot(gs[1, 1])
        strat_running_max = strat_cum.expanding().max()
        strat_dd = (strat_cum - strat_running_max) / strat_running_max
        bench_running_max = bench_cum.expanding().max()
        bench_dd = (bench_cum - bench_running_max) / bench_running_max

        ax3.fill_between(strat_dd.index, strat_dd.values, 0,
                        alpha=0.3, color='#2E86AB', label='Strategy')
        ax3.fill_between(bench_dd.index, bench_dd.values, 0,
                        alpha=0.3, color='#A23B72', label='Benchmark')
        ax3.set_title('Drawdown', fontsize=12, fontweight='bold')
        ax3.set_ylabel('Drawdown', fontsize=10)
        ax3.legend(fontsize=9)
        ax3.grid(True, alpha=0.3)

        # 4. Monthly Returns Heatmap
        ax4 = fig.add_subplot(gs[1, 2])
        monthly_returns = strat_ret.resample('ME').apply(lambda x: (1 + x).prod() - 1)
        monthly_returns_pivot = monthly_returns.groupby([
            monthly_returns.index.year,
            monthly_returns.index.month
        ]).first().unstack()

        if len(monthly_returns_pivot) > 0:
            sns.heatmap(monthly_returns_pivot * 100, annot=True, fmt='.1f',
                       cmap='RdYlGn', center=0, ax=ax4, cbar_kws={'label': 'Return (%)'})
            ax4.set_title('Monthly Returns (%)', fontsize=12, fontweight='bold')
            ax4.set_xlabel('Month', fontsize=10)
            ax4.set_ylabel('Year', fontsize=10)

        # 5. Return Distribution
        ax5 = fig.add_subplot(gs[2, 0])
        ax5.hist(strat_ret * 100, bins=50, alpha=0.7, color='#2E86AB',
                label='Strategy', density=True)
        ax5.hist(bench_ret * 100, bins=50, alpha=0.7, color='#A23B72',
                label='Benchmark', density=True)
        ax5.set_title('Return Distribution', fontsize=12, fontweight='bold')
        ax5.set_xlabel('Daily Return (%)', fontsize=10)
        ax5.set_ylabel('Density', fontsize=10)
        ax5.legend(fontsize=9)
        ax5.grid(True, alpha=0.3)
        ax5.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

        # 6. Performance Metrics Table
        ax6 = fig.add_subplot(gs[2, 1:])
        ax6.axis('tight')
        ax6.axis('off')

        metrics_data = [
            ['Metric', 'Strategy', 'Benchmark', 'Difference'],
            ['Total Return',
             f"{results['total_return_strategy']*100:.2f}%",
             f"{results['total_return_benchmark']*100:.2f}%",
             f"{(results['total_return_strategy']-results['total_return_benchmark'])*100:.2f}%"],
            ['CAGR',
             f"{results['cagr_strategy']*100:.2f}%",
             f"{results['cagr_benchmark']*100:.2f}%",
             f"{(results['cagr_strategy']-results['cagr_benchmark'])*100:.2f}%"],
            ['Volatility',
             f"{results['volatility_strategy']*100:.2f}%",
             f"{results['volatility_benchmark']*100:.2f}%",
             f"{(results['volatility_strategy']-results['volatility_benchmark'])*100:.2f}%"],
            ['Sharpe Ratio',
             f"{results['sharpe_strategy']:.2f}",
             f"{results['sharpe_benchmark']:.2f}",
             f"{results['sharpe_strategy']-results['sharpe_benchmark']:.2f}"],
            ['Sortino Ratio',
             f"{results['sortino_strategy']:.2f}",
             f"{results['sortino_benchmark']:.2f}",
             f"{results['sortino_strategy']-results['sortino_benchmark']:.2f}"],
            ['Max Drawdown',
             f"{results['max_drawdown_strategy']*100:.2f}%",
             f"{results['max_drawdown_benchmark']*100:.2f}%",
             f"{(results['max_drawdown_strategy']-results['max_drawdown_benchmark'])*100:.2f}%"],
            ['Win Rate',
             f"{results['win_rate_strategy']*100:.2f}%",
             f"{results['win_rate_benchmark']*100:.2f}%",
             f"{(results['win_rate_strategy']-results['win_rate_benchmark'])*100:.2f}%"],
        ]

        table = ax6.table(cellText=metrics_data, cellLoc='center', loc='center',
                         bbox=[0, 0, 1, 1])
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 2)

        for i in range(4):
            table[(0, i)].set_facecolor('#2E86AB')
            table[(0, i)].set_text_props(weight='bold', color='white')

        for i in range(1, len(metrics_data)):
            for j in range(4):
                if i % 2 == 0:
                    table[(i, j)].set_facecolor('#F0F0F0')

        plt.suptitle(f'Semi-Volatility Scaled Momentum Strategy - Performance Dashboard (FIXED)\n'
                    f'Period: {config.START_DATE} to {config.END_DATE}',
                    fontsize=16, fontweight='bold', y=0.98)

        plt.show()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function for the backtest."""
    config = StrategyConfig()

    backtester = SMOMBacktester(config)
    results = backtester.run_backtest()

    print("\n" + "="*80)
    print("PERFORMANCE SUMMARY")
    print("="*80)
    print(f"Strategy Total Return:    {results['total_return_strategy']*100:>8.2f}%")
    print(f"Benchmark Total Return:   {results['total_return_benchmark']*100:>8.2f}%")
    print(f"Outperformance:           {(results['total_return_strategy']-results['total_return_benchmark'])*100:>8.2f}%")
    print("-"*80)
    print(f"Strategy CAGR:            {results['cagr_strategy']*100:>8.2f}%")
    print(f"Benchmark CAGR:           {results['cagr_benchmark']*100:>8.2f}%")
    print("-"*80)
    print(f"Strategy Volatility:      {results['volatility_strategy']*100:>8.2f}%")
    print(f"Benchmark Volatility:     {results['volatility_benchmark']*100:>8.2f}%")
    print("-"*80)
    print(f"Strategy Sharpe Ratio:    {results['sharpe_strategy']:>8.2f}")
    print(f"Benchmark Sharpe Ratio:   {results['sharpe_benchmark']:>8.2f}")
    print("-"*80)
    print(f"Strategy Sortino Ratio:   {results['sortino_strategy']:>8.2f}")
    print(f"Benchmark Sortino Ratio:  {results['sortino_benchmark']:>8.2f}")
    print("-"*80)
    print(f"Strategy Max Drawdown:    {results['max_drawdown_strategy']*100:>8.2f}%")
    print(f"Benchmark Max Drawdown:   {results['max_drawdown_benchmark']*100:>8.2f}%")
    print("="*80)

    PerformanceVisualizer.plot_results(results, config)

    return results


if __name__ == "__main__":
    results = main()
